In [11]:
df = spark.read.format("csv").option("header","true").load("Files/products.csv")
display(df)

StatementMeta(, c4870a97-fc26-425b-9bde-187385b8d440, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f94dd66b-8b85-4ab9-b2f6-b06f190acc5c)

In [12]:
# Explore delta lake functionlaity

from pyspark.sql.types import StructType, IntegerType, StringType, DoubleType

# define schema

schema = StructType() \
.add("ProductID", IntegerType(), True) \
.add("ProductName", StringType(), True) \
.add("Category", StringType(), True) \
.add("ListPrice", DoubleType(), True)

df = spark.read.format("csv").option("header", "true").schema(schema).load("Files/products.csv")

display(df)


StatementMeta(, c4870a97-fc26-425b-9bde-187385b8d440, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5219753f-8adf-498f-a200-95ce5968dda5)

In [13]:
df.write.format("delta").saveAsTable("managed_products")


StatementMeta(, c4870a97-fc26-425b-9bde-187385b8d440, 16, Finished, Available, Finished, False)

In [14]:
df.write.format("delta").saveAsTable("external_products",path="abfss://DataEngineering@onelake.dfs.fabric.microsoft.com/Data.Lakehouse/Files/external_products")



StatementMeta(, c4870a97-fc26-425b-9bde-187385b8d440, 17, Finished, Available, Finished, False)

AnalysisException: [DELTA_CREATE_TABLE_WITH_NON_EMPTY_LOCATION] Cannot create table ('`chimcobldhq2ah31ehgkarj7d5n6apbid5n6e9a4c5q629b4c9ng`.`external_products`'). The associated location ('abfss://DataEngineering@onelake.dfs.fabric.microsoft.com/Data.Lakehouse/Files/external_products') is not empty and also not a Delta table.

In [1]:
%%sql
DESCRIBE FORMATTED managed_products;


StatementMeta(, c4870a97-fc26-425b-9bde-187385b8d440, 2, Finished, Available, Finished, False)

<Spark SQL result set with 11 rows and 3 fields>

In [2]:
%%sql
DESCRIBE FORMATTED external_products

StatementMeta(, c4870a97-fc26-425b-9bde-187385b8d440, 3, Finished, Available, Finished, False)

<Spark SQL result set with 11 rows and 3 fields>

In [9]:
%%sql
DROP TABLE managed_products;
DROP TABLE external_products;

StatementMeta(, c4870a97-fc26-425b-9bde-187385b8d440, 11, Finished, , Finished, True)

Error: [TABLE_OR_VIEW_NOT_FOUND] The table or view `spark_catalog`.`chimcobldhq2ah31ehgkarj7d5n6apbid5n6e9a4c5q629b4c9ng`.`managed_products` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.

In [7]:
%%sql
CREATE TABLE products
USING DELTA
LOCATION 'Files/external_products';

StatementMeta(, c4870a97-fc26-425b-9bde-187385b8d440, 9, Finished, Available, Finished, False)

Error: Invalid root directory, must be 'Files' or 'Tables', got: Tab

In [8]:
%%sql

SELECT * FROM products;

StatementMeta(, c4870a97-fc26-425b-9bde-187385b8d440, 10, Finished, Available, Finished, False)

Error: [TABLE_OR_VIEW_NOT_FOUND] The table or view `products` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 2 pos 14;
'Project [*]
+- 'UnresolvedRelation [products], [], false


In [ ]:
%%sql
UPDATE products
SET ListPrice = ListPrice * 0.9
WHERE Category = 'Furniture';



In [ ]:
%%sql
DESCRIBE HISTORY products;


In [ ]:
delta_table_path = 'Files/external_products'
current_data = spark.read.format("delta").load(delta_table_path)
display(current_data)

original_data = spark.read.format("delta").option("versionAsOf", 0).load(delta_table_path)
display(original_data)

## Analyse Delta table data using SQL queries

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW products_view
AS
    SELECT Category, Count(*) AS NumProducts, MIN(ListPrice) AS MinPrice, MAX(ListPrice) AS MaxPrice, AVG(ListPrice) AS AvgPrice
    FROM products
    GROUP BY Category;

SELECT * FROM products_view
ORDER BY Category;

In [ ]:
%%sql
SELECT Category, NumProducts
FROM products_view
ORDER BY NumProducts DESC
LIMIT 10;


In [ ]:
from pyspark.sql.functions import col, desc

df_products = spark.sql("SELECT Category, MinPrice, MaxPrice, AvgPrice FROM products_view").orderBy(col("AvgPrice").desc())
display(df_products.limits(6))

## USE Delta tables for Stream Data

In [15]:
from notebookutils import mssparkutils
from pyspark.sql.types import *
from pyspark.sql.functions import *

# create folder
inputPath = 'Files/data/'
mssparkutils.fs.mkdirs(inputPath)

# create a stream that reads data from the folder, using JSON Schema
jsonSchema = StructType([
    StructField("device", StringType(), False),
    StructField("status", StringType(), False)    
 ])

iotstream = spark.readStream.schema(jsonSchema).option("maxFilesPerTriger",1).json(inputPath)
# Write some event data to the folder

device_data ='''{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}
{"device":"Dev1","status":"error"}
{"device":"Dev2","status":"ok"}
{"device":"Dev2","status":"error"}
{"device":"Dev1","status":"ok"}'''

mssparkutils.fs.put(inputPath + "data.txt", device_data, True)

print("Source stream created...")

StatementMeta(, c4870a97-fc26-425b-9bde-187385b8d440, 18, Finished, Available, Finished, False)

Source stream created...
